# FRED API 全面教程

Federal Reserve Economic Data - 美联储经济数据


## 1. 安装


In [ ]:
!pip install fredapi pandas matplotlib seaborn


## 2. 初始化


In [ ]:
from fredapi import Fred
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 获取 API key: https://fred.stlouisfed.org/docs/api/api_key.html
fred = Fred(api_key='<your_api_key>')


## 3. 获取单个序列


In [ ]:
# GDP
gdp = fred.get_series('GDP')
print(gdp.tail())


## 4. 指定日期范围


In [ ]:
# 2020年至今的失业率
unrate = fred.get_series('UNRATE', observation_start='2020-01-01')
print(unrate.head())


## 5. 获取序列信息


In [ ]:
# 查看序列元数据
info = fred.get_series_info('GDP')
print(f"标题: {info['title']}")
print(f"单位: {info['units']}")
print(f"频率: {info['frequency']}")


## 6. 搜索序列


In [ ]:
# 搜索通胀相关数据
results = fred.search('inflation')
print(results[['id', 'title', 'frequency']].head(10))


## 7. 常用经济指标


In [ ]:
# 主要经济指标
indicators = {
    'GDP': 'GDP',                    # 国内生产总值
    'UNRATE': 'UNRATE',              # 失业率
    'CPIAUCSL': 'CPIAUCSL',          # CPI 消费者价格指数
    'FEDFUNDS': 'FEDFUNDS',          # 联邦基金利率
    'DGS10': 'DGS10',                # 10年期国债收益率
    'SP500': 'SP500',                # 标普500指数
    'DEXCHUS': 'DEXCHUS',            # 美元/人民币汇率
    'MORTGAGE30US': 'MORTGAGE30US'   # 30年抵押贷款利率
}

data = {name: fred.get_series(code, observation_start='2020-01-01') 
        for name, code in indicators.items()}
df = pd.DataFrame(data)
print(df.tail())


## 8. 数据可视化


In [ ]:
# 单个指标
plt.figure(figsize=(12, 6))
fred.get_series('UNRATE', observation_start='2010-01-01').plot()
plt.title('美国失业率 (2010-至今)')
plt.ylabel('百分比')
plt.grid(True)
plt.show()


## 9. 多指标对比


In [ ]:
# 利率对比
fig, ax = plt.subplots(figsize=(14, 7))
fred.get_series('FEDFUNDS', observation_start='2015-01-01').plot(ax=ax, label='联邦基金利率')
fred.get_series('DGS10', observation_start='2015-01-01').plot(ax=ax, label='10年期国债')
fred.get_series('MORTGAGE30US', observation_start='2015-01-01').plot(ax=ax, label='30年抵押贷款')
plt.title('美国利率走势')
plt.ylabel('百分比')
plt.legend()
plt.grid(True)
plt.show()


## 10. 数据转换


In [ ]:
# 计算同比增长率
cpi = fred.get_series('CPIAUCSL', observation_start='2020-01-01')
inflation = cpi.pct_change(periods=12) * 100  # 年度通胀率

plt.figure(figsize=(12, 6))
inflation.plot()
plt.title('美国通胀率 (同比)')
plt.ylabel('百分比')
plt.axhline(y=2, color='r', linestyle='--', label='2% 目标')
plt.legend()
plt.grid(True)
plt.show()


## 11. 相关性分析


In [ ]:
# 多指标相关性
data = pd.DataFrame({
    '失业率': fred.get_series('UNRATE', observation_start='2015-01-01'),
    '联邦基金利率': fred.get_series('FEDFUNDS', observation_start='2015-01-01'),
    '标普500': fred.get_series('SP500', observation_start='2015-01-01')
})

plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('经济指标相关性')
plt.show()


## 12. 导出数据


In [ ]:
# 保存为 CSV
df.to_csv('fred_data.csv')

# 保存为 Excel
df.to_excel('fred_data.xlsx')


## 13. 实用函数


In [ ]:
def get_latest_value(series_id):
    """获取最新值"""
    data = fred.get_series(series_id)
    return data.iloc[-1]

def get_yoy_change(series_id):
    """计算同比变化"""
    data = fred.get_series(series_id)
    return data.pct_change(periods=12).iloc[-1] * 100

# 使用示例
print(f"当前失业率: {get_latest_value('UNRATE'):.2f}%")
print(f"CPI同比: {get_yoy_change('CPIAUCSL'):.2f}%")
